In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  02_limpeza.ipynb — Limpeza, enriquecimento e unificação   ║
# ║  TCC: Cobertura Vacinal no Piauí (2015–2023)               ║
# ║  Dani Alcoforado — Tecnólogo em Ciência de Dados (UNINTER) ║
# ╚══════════════════════════════════════════════════════════════╝
#
# ORDEM DE EXECUÇÃO:
#   1. Célula 0  → instalar dependências (reiniciar após)
#   2. Célula 1  → montar Drive e definir caminhos
#   3. Células 2–5  → enriquecer dados PySUS (2015–2019)
#   4. Células 6–9  → carregar dados TabNet (2020–2023)
#   5. Células 10–12 → unificar e validar dataset 2015–2023
#   6. Célula 13 → salvar artefatos finais
#
# PRÉ-REQUISITO: 01_coleta.ipynb concluído
#   dados_tratados/pni_piaui_clean.parquet  ← input principal
#   dados_brutos/dicionario_vacinas_sipni.csv

In [ ]:
# ── CÉLULA 0: Instalar dependências ───────────────────────────
# ⚠️ Rode esta célula, depois:
#    Ambiente de execução > Reiniciar ambiente
#    Continue a partir da Célula 1

!pip install openpyxl --quiet   # leitura de .xlsx do TabNet
!pip install unidecode --quiet  # normalização de nomes de municípios
print("✅ Dependências instaladas — reinicie o ambiente antes de continuar.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.0 MB/s eta 0:00:00
✅ Dependências instaladas — reinicie o ambiente antes de continuar.


In [ ]:
# ── CÉLULA 1: Montar Drive e definir caminhos ─────────────────

from google.colab import drive
drive.mount('/content/drive')

import os
import warnings
warnings.filterwarnings('ignore')

RAIZ           = '/content/drive/MyDrive/TCC_Vacinal_Piaui'
DADOS_BRUTOS   = f'{RAIZ}/dados_brutos'
DADOS_TRATADOS = f'{RAIZ}/dados_tratados'
FIGURAS        = f'{RAIZ}/figuras'

for pasta in [DADOS_BRUTOS, DADOS_TRATADOS, FIGURAS]:
    os.makedirs(pasta, exist_ok=True)

print("✅ Drive montado e pastas prontas!")

# ── Verificar arquivos gerados no notebook anterior ───────────
arqs_esperados = [
    f'{DADOS_TRATADOS}/pni_piaui_clean.parquet',
    f'{DADOS_TRATADOS}/pni_parnaiba_clean.parquet',
    f'{DADOS_BRUTOS}/dicionario_vacinas_sipni.csv',
]
print("\n📂 Verificando arquivos do 01_coleta:")
for arq in arqs_esperados:
    existe = os.path.exists(arq)
    status = '✅' if existe else '❌ FALTANDO'
    tam    = f"({os.path.getsize(arq)/1024:.0f} KB)" if existe else ''
    print(f"  {status}  {os.path.basename(arq)} {tam}")

Mounted at /content/drive
✅ Drive montado e pastas prontas!

📂 Verificando arquivos do 01_coleta:
  ✅  pni_piaui_clean.parquet (3424 KB)
  ✅  pni_parnaiba_clean.parquet (71 KB)
  ✅  dicionario_vacinas_sipni.csv (2 KB)


In [ ]:
# ── CÉLULA 2: Carregar dataset limpo do 01_coleta ─────────────

import pandas as pd
import numpy as np

df = pd.read_parquet(f'{DADOS_TRATADOS}/pni_piaui_clean.parquet')

print(f"Dataset carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print("\nColunas:", df.columns.tolist())
print("\nAnos disponíveis:", sorted(df['ano'].unique().tolist()))
print("\nTipos:")
print(df.dtypes)

Dataset carregado: 855,767 linhas × 16 colunas

Colunas: ['ano', 'UF', 'cod_municipio', 'cod_vacina', 'doses_aplicadas', 'populacao_alvo', 'cobertura_pct', 'ANOMES', 'mes', 'cod_faixa', 'DOSE', 'DOSE1', 'DOSEN', 'DIFER', 'vacina_nome', 'faixa_nome']

Anos disponíveis: [2015, 2016, 2017, 2018, 2019]

Tipos:
ano                  int64
UF                  object
cod_municipio       object
cod_vacina          object
doses_aplicadas      int64
populacao_alvo     float64
cobertura_pct      float64
ANOMES              object
mes                float64
cod_faixa           object
DOSE                object
DOSE1               object
DOSEN               object
DIFER               object
vacina_nome         object
faixa_nome          object
dtype: object


In [ ]:
# ── CÉLULA 3 DEFINITIVA: Municípios do Piauí (Censo 2010 IBGE) ─
# Regra: cod_municipio = 6 primeiros dígitos do código IBGE de 7 dígitos
# Fonte: IBGE, Censo Demográfico 2010 — lista oficial dos 224 municípios

MUNICIPIOS_PI = {
    '220005': 'Acauã',
    '220010': 'Agricolândia',
    '220020': 'Água Branca',
    '220025': 'Alagoinha do Piauí',
    '220027': 'Alegrete do Piauí',
    '220030': 'Alto Longá',
    '220040': 'Altos',
    '220045': 'Alvorada do Gurguéia',
    '220050': 'Amarante',
    '220060': 'Angical do Piauí',
    '220070': 'Anísio de Abreu',
    '220080': 'Antônio Almeida',
    '220090': 'Aroazes',
    '220095': 'Aroeiras do Itaim',
    '220100': 'Arraial',
    '220105': 'Assunção do Piauí',
    '220110': 'Avelino Lopes',
    '220115': 'Baixa Grande do Ribeiro',
    '220117': "Barra D'Alcântara",
    '220120': 'Barras',
    '220130': 'Barreiras do Piauí',
    '220140': 'Barro Duro',
    '220150': 'Batalha',
    '220155': 'Bela Vista do Piauí',
    '220157': 'Belém do Piauí',
    '220160': 'Beneditinos',
    '220170': 'Bertolínia',
    '220173': 'Betânia do Piauí',
    '220177': 'Boa Hora',
    '220180': 'Bocaina',
    '220190': 'Bom Jesus',
    '220191': 'Bom Princípio do Piauí',
    '220192': 'Bonfim do Piauí',
    '220194': 'Boqueirão do Piauí',
    '220196': 'Brasileira',
    '220198': 'Brejo do Piauí',
    '220200': 'Buriti dos Lopes',
    '220202': 'Buriti dos Montes',
    '220205': 'Cabeceiras do Piauí',
    '220207': 'Cajazeiras do Piauí',
    '220208': 'Cajueiro da Praia',
    '220209': 'Caldeirão Grande do Piauí',
    '220210': 'Campinas do Piauí',
    '220211': 'Campo Alegre do Fidalgo',
    '220213': 'Campo Grande do Piauí',
    '220217': 'Campo Largo do Piauí',
    '220220': 'Campo Maior',
    '220225': 'Canavieira',
    '220230': 'Canto do Buriti',
    '220240': 'Capitão de Campos',
    '220245': 'Capitão Gervásio Oliveira',
    '220250': 'Caracol',
    '220253': 'Caraúbas do Piauí',
    '220255': 'Caridade do Piauí',
    '220260': 'Castelo do Piauí',
    '220265': 'Caxingó',
    '220270': 'Cocal',
    '220271': 'Cocal de Telha',
    '220272': 'Cocal dos Alves',
    '220273': 'Coivaras',
    '220275': 'Colônia do Gurguéia',
    '220277': 'Colônia do Piauí',
    '220280': 'Conceição do Canindé',
    '220285': 'Coronel José Dias',
    '220290': 'Corrente',
    '220300': 'Cristalândia do Piauí',
    '220310': 'Cristino Castro',
    '220320': 'Curimatá',
    '220323': 'Currais',
    '220325': 'Curralinhos',
    '220327': 'Curral Novo do Piauí',
    '220330': 'Demerval Lobão',
    '220335': 'Dirceu Arcoverde',
    '220340': 'Dom Expedito Lopes',
    '220342': 'Domingos Mourão',
    '220345': 'Dom Inocêncio',
    '220350': 'Elesbão Veloso',
    '220360': 'Eliseu Martins',
    '220370': 'Esperantina',
    '220375': 'Fartura do Piauí',
    '220380': 'Flores do Piauí',
    '220385': 'Floresta do Piauí',
    '220390': 'Floriano',
    '220400': 'Francinópolis',
    '220410': 'Francisco Ayres',
    '220415': 'Francisco Macedo',
    '220420': 'Francisco Santos',
    '220430': 'Fronteiras',
    '220435': 'Geminiano',
    '220440': 'Gilbués',
    '220450': 'Guadalupe',
    '220455': 'Guaribas',
    '220460': 'Hugo Napoleão',
    '220465': 'Ilha Grande',
    '220470': 'Inhuma',
    '220480': 'Ipiranga do Piauí',
    '220490': 'Isaías Coelho',
    '220500': 'Itainópolis',
    '220510': 'Itaueira',
    '220515': 'Jacobina do Piauí',
    '220520': 'Jaicós',
    '220525': 'Jardim do Mulato',
    '220527': 'Jatobá do Piauí',
    '220530': 'Jerumenha',
    '220535': 'João Costa',
    '220540': 'Joaquim Pires',
    '220545': 'Joca Marques',
    '220550': 'José de Freitas',
    '220551': 'Juazeiro do Piauí',
    '220552': 'Júlio Borges',
    '220553': 'Jurema',
    '220554': 'Lagoinha do Piauí',
    '220555': 'Lagoa Alegre',
    '220556': 'Lagoa do Barro do Piauí',
    '220557': 'Lagoa de São Francisco',
    '220558': 'Lagoa do Piauí',
    '220559': 'Lagoa do Sítio',
    '220560': 'Landri Sales',
    '220570': 'Luís Correia',
    '220580': 'Luzilândia',
    '220585': 'Madeiro',
    '220590': 'Manoel Emídio',
    '220595': 'Marcolândia',
    '220600': 'Marcos Parente',
    '220605': 'Massapê do Piauí',
    '220610': 'Matias Olímpio',
    '220620': 'Miguel Alves',
    '220630': 'Miguel Leão',
    '220635': 'Milton Brandão',
    '220640': 'Monsenhor Gil',
    '220650': 'Monsenhor Hipólito',
    '220660': 'Monte Alegre do Piauí',
    '220665': 'Morro Cabeça no Tempo',
    '220667': 'Morro do Chapéu do Piauí',
    '220669': 'Murici dos Portelas',
    '220670': 'Nazaré do Piauí',
    '220672': 'Nazária',
    '220675': 'Nossa Senhora de Nazaré',
    '220680': 'Nossa Senhora dos Remédios',
    '220690': 'Novo Oriente do Piauí',
    '220695': 'Novo Santo Antônio',
    '220700': 'Oeiras',
    '220710': "Olho D'Água do Piauí",
    '220720': 'Padre Marcos',
    '220730': 'Paes Landim',
    '220735': 'Pajeú do Piauí',
    '220740': 'Palmeira do Piauí',
    '220750': 'Palmeirais',
    '220755': 'Paquetá',
    '220760': 'Parnaguá',
    '220770': 'Parnaíba',
    '220775': 'Passagem Franca do Piauí',
    '220777': 'Patos do Piauí',
    '220779': "Pau D'Arco do Piauí",
    '220780': 'Paulistana',
    '220785': 'Pavussu',
    '220790': 'Pedro II',
    '220793': 'Pedro Laurentino',
    '220795': 'Nova Santa Rita',
    '220800': 'Picos',
    '220810': 'Pimenteiras',
    '220820': 'Pio IX',
    '220830': 'Piracuruca',
    '220840': 'Piripiri',
    '220850': 'Porto',
    '220855': 'Porto Alegre do Piauí',
    '220860': 'Prata do Piauí',
    '220865': 'Queimada Nova',
    '220870': 'Redenção do Gurguéia',
    '220880': 'Regeneração',
    '220885': 'Riacho Frio',
    '220887': 'Ribeira do Piauí',
    '220890': 'Ribeiro Gonçalves',
    '220900': 'Rio Grande do Piauí',
    '220910': 'Santa Cruz do Piauí',
    '220915': 'Santa Cruz dos Milagres',
    '220920': 'Santa Filomena',
    '220930': 'Santa Luz',
    '220935': 'Santana do Piauí',
    '220937': 'Santa Rosa do Piauí',
    '220940': 'Santo Antônio de Lisboa',
    '220945': 'Santo Antônio dos Milagres',
    '220950': 'Santo Inácio do Piauí',
    '220955': 'São Braz do Piauí',
    '220960': 'São Félix do Piauí',
    '220965': 'São Francisco de Assis do Piauí',
    '220970': 'São Francisco do Piauí',
    '220975': 'São Gonçalo do Gurguéia',
    '220980': 'São Gonçalo do Piauí',
    '220985': 'São João da Canabrava',
    '220987': 'São João da Fronteira',
    '220990': 'São João da Serra',
    '220995': 'São João da Varjota',
    '220997': 'São João do Arraial',
    '221000': 'São João do Piauí',
    '221005': 'São José do Divino',
    '221010': 'São José do Peixe',
    '221020': 'São José do Piauí',
    '221030': 'São Julião',
    '221035': 'São Lourenço do Piauí',
    '221037': 'São Luis do Piauí',
    '221038': 'São Miguel da Baixa Grande',
    '221039': 'São Miguel do Fidalgo',
    '221040': 'São Miguel do Tapuio',
    '221050': 'São Pedro do Piauí',
    '221060': 'São Raimundo Nonato',
    '221062': 'Sebastião Barros',
    '221063': 'Sebastião Leal',
    '221065': 'Sigefredo Pacheco',
    '221070': 'Simões',
    '221080': 'Simplício Mendes',
    '221090': 'Socorro do Piauí',
    '221093': 'Sussuapara',
    '221095': 'Tamboril do Piauí',
    '221097': 'Tanque do Piauí',
    '221100': 'Teresina',
    '221110': 'União',
    '221120': 'Uruçuí',
    '221130': 'Valença do Piauí',
    '221135': 'Várzea Branca',
    '221140': 'Várzea Grande',
    '221150': 'Vera Mendes',
    '221160': 'Vila Nova do Piauí',
    '221170': 'Wall Ferraz',
}

# Aplicar ao dataset
df['nome_municipio'] = df['cod_municipio'].astype(str).str.strip().map(MUNICIPIOS_PI)

# Diagnóstico
mapeados     = df['nome_municipio'].notna().sum()
nao_mapeados = df[df['nome_municipio'].isna()]['cod_municipio'].value_counts()
print(f"Municípios mapeados: {mapeados:,} de {len(df):,} ({mapeados/len(df)*100:.1f}%)")

if len(nao_mapeados) > 0:
    print(f"\nCódigos ainda não mapeados ({len(nao_mapeados)}):")
    print(nao_mapeados.head(20))
else:
    print("\n✅ Todos os municípios mapeados!")

# Conferir Parnaíba
parnaiba_check = df[df['cod_municipio'] == '220770']
print(f"\nLinhas de Parnaíba (220770): {len(parnaiba_check):,}")

Municípios mapeados: 855,767 de 855,767 (100.0%)

✅ Todos os municípios mapeados!

Linhas de Parnaíba (220770): 14,204


In [ ]:
# ── CÉLULA 4 CORRIGIDA: Mapeamento completo de faixas etárias ─

FAIXAS_COMPLETO = {
    # ── Infantil < 1 ano ──────────────────────────────────────
    '50':  'Menor de 1 ano',
    'F5':  'Feminino < 1 ano',
    'G5':  'Masculino < 1 ano',
    'F6':  'Feminino 6 meses',
    'G6':  'Masculino 6 meses',
    'F9':  'Feminino 9 meses',
    'G9':  'Masculino 9 meses',

    # ── Infantil 1–4 anos ─────────────────────────────────────
    '51':  '1 ano',
    '52':  '2 anos',
    '53':  '3 anos',
    '54':  '4 anos',
    'F1':  'Feminino 1 ano',
    'G1':  'Masculino 1 ano',
    'F2':  'Feminino 2 anos',
    'G2':  'Masculino 2 anos',
    'F3':  'Feminino 3 anos',
    'G3':  'Masculino 3 anos',
    'F4':  'Feminino 4 anos',
    'G4':  'Masculino 4 anos',

    # ── Escolar 5–9 anos ──────────────────────────────────────
    '55':  '5 anos',
    '56':  '6 anos',
    '57':  '7 anos',
    '58':  '8 anos',
    '59':  '9 anos',
    'F7':  'Feminino 7 anos',
    'G7':  'Masculino 7 anos',
    'F8':  'Feminino 8 anos',
    'G8':  'Masculino 8 anos',
    # Faixas por série escolar
    'S1':  'Escolar — 1º ano',
    'S2':  'Escolar — 2º ano',
    'S3':  'Escolar — 3º ano',
    'S4':  'Escolar — 4º ano',
    'S5':  'Escolar — 5º ano',
    'S6':  'Escolar — 6º ano',
    'S7':  'Escolar — 7º ano',
    'S8':  'Escolar — 8º ano',
    'S9':  'Escolar — 9º ano',
    # Faixas por ciclo
    'C4':  'Ciclo 4',
    'C5':  'Ciclo 5',
    'C6':  'Ciclo 6',
    'C7':  'Ciclo 7',
    'C8':  'Ciclo 8',

    # ── Adolescente 10–19 anos ────────────────────────────────
    '61':  'Adolescente (10–19 anos)',
    '01':  'Adolescente',
    '03':  'Adolescente (12–13 anos)',
    '16':  'Adolescente (16 anos)',
    # Faixas X = campanha HPV/adolescente (por faixa etária de campanha)
    'X1':  'Campanha — 9 anos',
    'X2':  'Campanha — 10 anos',
    'X3':  'Campanha — 11 anos',
    'X4':  'Campanha — 12 anos',
    'X5':  'Campanha — 13 anos',
    'X6':  'Campanha — 14 anos',
    'X7':  'Campanha — 15 anos',

    # ── Adulto 20–59 anos ─────────────────────────────────────
    '17':  'Adulto (17–29 anos)',
    '18':  'Adulto (18–29 anos)',
    '72':  'Adulto Trabalhador',
    '74':  'Adulto (20–59 anos)',
    '75':  'Adulto (30–59 anos)',
    # Faixas D = adulto por faixa (campanha dT/influenza adulto)
    'D1':  'Adulto (20–29 anos)',
    'D2':  'Adulto (30–39 anos)',
    'D3':  'Adulto (40–49 anos)',
    'D4':  'Adulto (50–59 anos)',
    'D5':  'Adulto (20–59 anos)',
    'D6':  'Adulto (25–29 anos)',
    'D7':  'Adulto (30–59 anos)',
    'D8':  'Adulto (50–59 anos)',
    'D9':  'Adulto (18–59 anos)',

    # ── Gestante / Puérpera ───────────────────────────────────
    'G':   'Gestante',
    'P':   'Puérpera',

    # ── Idoso 60+ anos ────────────────────────────────────────
    '32':  'Idoso (≥ 60 anos)',
    '97':  'Idoso (60+ anos)',
    'R2':  'Idoso (60–69 anos)',
    'R3':  'Idoso (70–79 anos)',
    'R4':  'Idoso (80+ anos)',
    'R5':  'Idoso (60–64 anos)',
    'R6':  'Idoso (65–69 anos)',
    'R7':  'Idoso (70–74 anos)',
    'R8':  'Idoso (75–79 anos)',
    'R9':  'Idoso (80+ anos)',

    # ── Indígena ──────────────────────────────────────────────
    'A4':  'Indígena < 1 ano',
    'A8':  'Indígena adulto',

    # Faixas numéricas alternativas
    '04':  '4 anos',
    '09':  '9 anos',
    '20':  'Adulto (20–29 anos)',
    '31':  'Adulto (30–39 anos)',
    '64':  'Idoso (60–64 anos)',
    '67':  'Idoso (65–69 anos)',
    '68':  'Idoso (60–69 anos)',
    '69':  'Idoso (60–69 anos)',
    '73':  'Idoso (70–74 anos)',
    '79':  'Idoso (70–79 anos)',
    '87':  'Idoso (80+ anos)',
    # Faixas alfabéticas adicionais
    'A0':  'Indígena < 1 ano',
    'A1':  'Indígena 1 ano',
    'E0':  'Escolar — pré-escola',
    'E1':  'Escolar — 1º ano (especial)',
    'R0':  'Adulto (50–59 anos)',
    'R1':  'Adulto (55–59 anos)',
    'S0':  'Escolar — pré-escola',
    'W1':  'Campanha — adulto faixa 1',
    'W2':  'Campanha — adulto faixa 2',
}

# ── Grupo etário simplificado para análise ────────────────────
GRUPO_ETARIO = {
    # Infantil < 1 ano
    'Menor de 1 ano':         'Infantil (< 1 ano)',
    'Feminino < 1 ano':       'Infantil (< 1 ano)',
    'Masculino < 1 ano':      'Infantil (< 1 ano)',
    'Feminino 6 meses':       'Infantil (< 1 ano)',
    'Masculino 6 meses':      'Infantil (< 1 ano)',
    'Feminino 9 meses':       'Infantil (< 1 ano)',
    'Masculino 9 meses':      'Infantil (< 1 ano)',
    # Infantil 1–4 anos
    '1 ano':                  'Infantil (1–4 anos)',
    '2 anos':                 'Infantil (1–4 anos)',
    '3 anos':                 'Infantil (1–4 anos)',
    '4 anos':                 'Infantil (1–4 anos)',
    'Feminino 1 ano':         'Infantil (1–4 anos)',
    'Masculino 1 ano':        'Infantil (1–4 anos)',
    'Feminino 2 anos':        'Infantil (1–4 anos)',
    'Masculino 2 anos':       'Infantil (1–4 anos)',
    'Feminino 3 anos':        'Infantil (1–4 anos)',
    'Masculino 3 anos':       'Infantil (1–4 anos)',
    'Feminino 4 anos':        'Infantil (1–4 anos)',
    'Masculino 4 anos':       'Infantil (1–4 anos)',
    # Escolar 5–9 anos
    '5 anos':                 'Escolar (5–9 anos)',
    '6 anos':                 'Escolar (5–9 anos)',
    '7 anos':                 'Escolar (5–9 anos)',
    '8 anos':                 'Escolar (5–9 anos)',
    '9 anos':                 'Escolar (5–9 anos)',
    'Feminino 7 anos':        'Escolar (5–9 anos)',
    'Masculino 7 anos':       'Escolar (5–9 anos)',
    'Feminino 8 anos':        'Escolar (5–9 anos)',
    'Masculino 8 anos':       'Escolar (5–9 anos)',
    'Escolar — 1º ano':       'Escolar (5–9 anos)',
    'Escolar — 2º ano':       'Escolar (5–9 anos)',
    'Escolar — 3º ano':       'Escolar (5–9 anos)',
    'Escolar — 4º ano':       'Escolar (5–9 anos)',
    'Escolar — 5º ano':       'Escolar (5–9 anos)',
    'Escolar — 6º ano':       'Escolar (5–9 anos)',
    'Escolar — 7º ano':       'Escolar (5–9 anos)',
    'Escolar — 8º ano':       'Escolar (5–9 anos)',
    'Escolar — 9º ano':       'Escolar (5–9 anos)',
    'Ciclo 4':                'Escolar (5–9 anos)',
    'Ciclo 5':                'Escolar (5–9 anos)',
    'Ciclo 6':                'Escolar (5–9 anos)',
    'Ciclo 7':                'Escolar (5–9 anos)',
    'Ciclo 8':                'Escolar (5–9 anos)',
    # Adolescente
    'Adolescente (10–19 anos)': 'Adolescente (10–19 anos)',
    'Adolescente':            'Adolescente (10–19 anos)',
    'Adolescente (12–13 anos)': 'Adolescente (10–19 anos)',
    'Adolescente (16 anos)':  'Adolescente (10–19 anos)',
    'Campanha — 9 anos':      'Escolar (5–9 anos)',
    'Campanha — 10 anos':     'Adolescente (10–19 anos)',
    'Campanha — 11 anos':     'Adolescente (10–19 anos)',
    'Campanha — 12 anos':     'Adolescente (10–19 anos)',
    'Campanha — 13 anos':     'Adolescente (10–19 anos)',
    'Campanha — 14 anos':     'Adolescente (10–19 anos)',
    'Campanha — 15 anos':     'Adolescente (10–19 anos)',
    # Adulto
    'Gestante':               'Adulto / Gestante',
    'Puérpera':               'Adulto / Gestante',
    'Adulto (17–29 anos)':    'Adulto (20–59 anos)',
    'Adulto (18–29 anos)':    'Adulto (20–59 anos)',
    'Adulto Trabalhador':     'Adulto (20–59 anos)',
    'Adulto (20–59 anos)':    'Adulto (20–59 anos)',
    'Adulto (30–59 anos)':    'Adulto (20–59 anos)',
    'Adulto (20–29 anos)':    'Adulto (20–59 anos)',
    'Adulto (30–39 anos)':    'Adulto (20–59 anos)',
    'Adulto (40–49 anos)':    'Adulto (20–59 anos)',
    'Adulto (50–59 anos)':    'Adulto (20–59 anos)',
    'Adulto (25–29 anos)':    'Adulto (20–59 anos)',
    'Adulto (18–59 anos)':    'Adulto (20–59 anos)',
    # Idoso
    'Idoso (≥ 60 anos)':      'Idoso (60+ anos)',
    'Idoso (60+ anos)':       'Idoso (60+ anos)',
    'Idoso (60–69 anos)':     'Idoso (60+ anos)',
    'Idoso (70–79 anos)':     'Idoso (60+ anos)',
    'Idoso (80+ anos)':       'Idoso (60+ anos)',
    'Idoso (60–64 anos)':     'Idoso (60+ anos)',
    'Idoso (65–69 anos)':     'Idoso (60+ anos)',
    'Idoso (70–74 anos)':     'Idoso (60+ anos)',
    'Idoso (75–79 anos)':     'Idoso (60+ anos)',
    # Indígena
    'Indígena < 1 ano':       'Indígena',
    'Indígena adulto':        'Indígena',

    # ── Adicionar ao GRUPO_ETARIO ──────────────────────────────────

    '4 anos':                  'Infantil (1–4 anos)',
    '9 anos':                  'Escolar (5–9 anos)',
    'Adulto (20–29 anos)':     'Adulto (20–59 anos)',
    'Adulto (30–39 anos)':     'Adulto (20–59 anos)',
    'Idoso (60–64 anos)':      'Idoso (60+ anos)',
    'Idoso (65–69 anos)':      'Idoso (60+ anos)',
    'Idoso (60–69 anos)':      'Idoso (60+ anos)',
    'Idoso (70–74 anos)':      'Idoso (60+ anos)',
    'Idoso (70–79 anos)':      'Idoso (60+ anos)',
    'Idoso (80+ anos)':        'Idoso (60+ anos)',
    'Indígena < 1 ano':        'Indígena',
    'Indígena 1 ano':          'Indígena',
    'Escolar — pré-escola':    'Escolar (5–9 anos)',
    'Escolar — 1º ano (especial)': 'Escolar (5–9 anos)',
    'Adulto (50–59 anos)':     'Adulto (20–59 anos)',
    'Adulto (55–59 anos)':     'Adulto (20–59 anos)',
    'Campanha — adulto faixa 1': 'Adulto (20–59 anos)',
    'Campanha — adulto faixa 2': 'Adulto (20–59 anos)',
}


# Aplicar mapeamentos
df['faixa_nome']  = df['cod_faixa'].astype(str).str.strip().map(FAIXAS_COMPLETO)
df['grupo_etario'] = df['faixa_nome'].map(GRUPO_ETARIO)

# Diagnóstico
mapeadas_f = df['faixa_nome'].notna().sum()
total = len(df)
print(f"=== Faixas mapeadas ===")
print(f"  {mapeadas_f:,} de {total:,} ({mapeadas_f/total*100:.1f}%)")

print("\n=== Distribuição por grupo etário ===")
print(df['grupo_etario'].value_counts())

sem_faixa = df[df['faixa_nome'].isna()]['cod_faixa'].value_counts()
print(f"\n=== Faixas ainda sem mapeamento ===")
print(sem_faixa.head(20) if len(sem_faixa) > 0 else '✅ Todas mapeadas!')

=== Faixas mapeadas ===
  823,746 de 855,767 (96.3%)

=== Distribuição por grupo etário ===
grupo_etario
Infantil (< 1 ano)          278389
Infantil (1–4 anos)         173325
Escolar (5–9 anos)          149436
Adolescente (10–19 anos)     87979
Idoso (60+ anos)             66856
Adulto (20–59 anos)          62793
Indígena                      4968
Name: count, dtype: int64

=== Faixas ainda sem mapeamento ===
cod_faixa
80    1026
94    1026
B5     787
27     759
15     653
A9     621
13     562
14     561
B6     552
C9     500
B4     451
B3     405
22     385
A5     300
E8     215
A2     207
E9     194
E7     156
E6     142
H1     137
Name: count, dtype: int64


In [ ]:
# ── CÉLULA 5: Tratar cobertura_pct anômala (> 150%) ──────────
# Valores > 100% são possíveis no SI-PNI quando campanhas vacinais
# captam pessoas de fora do município (denominador IBGE subestimado).
# DATASUS documenta esse comportamento. Estratégia:
#   • Manter valores originais — NÃO remover, pois são reais
#   • Criar flag 'cobertura_anomala' para análise
#   • Criar coluna 'cobertura_cap' capeada em 100% para visualização

# Diagnóstico antes do tratamento
print("=== Distribuição de cobertura_pct ===")
print(df['cobertura_pct'].describe().round(2))

print("\n=== Faixas de cobertura ===")
faixas_cob = pd.cut(
    df['cobertura_pct'].dropna(),
    bins=[-0.01, 0, 50, 95, 100, 150, float('inf')],
    labels=['Zero', '1–50%', '51–95%', '96–100%', '101–150%', '>150%']
)
print(faixas_cob.value_counts().sort_index())

# Criar colunas de tratamento
df['cobertura_anomala'] = df['cobertura_pct'] > 100
df['cobertura_cap']     = df['cobertura_pct'].clip(upper=100)
df['abaixo_meta']       = df['cobertura_pct'] < 95
df['critico']           = df['cobertura_pct'] < 50

# Percentual de anomalias
n_anom = df['cobertura_anomala'].sum()
n_cob  = df['cobertura_pct'].notna().sum()
print(f"\nRegistros com cobertura > 100%: {n_anom:,} de {n_cob:,} ({n_anom/n_cob*100:.1f}%)")
print("Nota: mantidos no dataset com flag 'cobertura_anomala' = True")

=== Distribuição de cobertura_pct ===
count    21443.00
mean        58.56
std         41.72
min          0.00
25%         21.43
50%         59.14
75%         89.76
max        342.86
Name: cobertura_pct, dtype: float64

=== Faixas de cobertura ===
cobertura_pct
Zero        1078
1–50%       8504
51–95%      7553
96–100%      981
101–150%    2951
>150%        376
Name: count, dtype: int64

Registros com cobertura > 100%: 3,327 de 21,443 (15.5%)
Nota: mantidos no dataset com flag 'cobertura_anomala' = True


In [ ]:
# ── CÉLULA 7 REVISADA: Carregar CSVs de doses por faixa (2020–2023) ──
# Formato: doses aplicadas por município × faixa etária (todas as vacinas)
# Fonte: TabNet DATASUS → dpnibr.def

import pandas as pd
import numpy as np
import os
import glob

# ── Faixas do calendário infantil disponíveis nesse formato ───
FAIXAS_INFANTIL = [
    'Ate 30 dias', '2 meses', '3 meses', '4 meses',
    '5 meses', '6 meses', '7 meses', '6 a 8 meses',
    'Menor de 1 ano', '1 ano', '2 anos', '3 anos', '4 anos',
]
FAIXAS_MENOR1  = ['Ate 30 dias','2 meses','3 meses','4 meses',
                   '5 meses','6 meses','7 meses','6 a 8 meses','Menor de 1 ano']
FAIXAS_1A4     = ['1 ano','2 anos','3 anos','4 anos']
FAIXAS_ADOLESC = ['9 anos','10 anos','11 anos','12 anos','13 anos',
                  '14 anos','9 a 12 anos','10 a 14 anos','9 a 19 anos']
FAIXAS_IDOSO   = ['60 a 64 anos','65 a 69 anos','70 a 74 anos',
                  '75 a 79 anos','60 anos e mais','65 anos e mais']


def carregar_tabnet_doses(caminho_csv, ano):
    """
    Lê CSV do TabNet (dpnibr.def) com doses por município × faixa etária.
    Retorna DataFrame com uma linha por município, colunas de doses agregadas.
    """
    # Detectar encoding
    for enc in ['latin-1', 'utf-8', 'cp1252']:
        try:
            with open(caminho_csv, encoding=enc) as f:
                linhas = f.readlines()
            break
        except:
            continue

    # Encontrar linha do header (contém 'Município' ou 'Munic')
    inicio = 0
    for i, linha in enumerate(linhas):
        if 'Munic' in linha or 'munic' in linha:
            inicio = i
            break

    # Encontrar rodapé (linha 'Total' ou vazia)
    fim = len(linhas)
    for i in range(inicio + 1, len(linhas)):
        linha = linhas[i].strip().strip('"')
        if linha.startswith('Total') or linha == '':
            fim = i
            break

    from io import StringIO
    bloco = ''.join(linhas[inicio:fim])
    df_raw = pd.read_csv(
        StringIO(bloco),
        sep=';',
        encoding='latin-1',
        dtype=str,
        quotechar='"'
    )

    # Limpar nomes de colunas
    df_raw.columns = [c.strip().strip('"').strip() for c in df_raw.columns]

    # Coluna município: primeira coluna
    col_mun = df_raw.columns[0]

    # Extrair cod_municipio (6 dígitos) e nome do município
    df_raw['cod_municipio'] = (
        df_raw[col_mun]
        .astype(str)
        .str.extract(r'(\d{6})')
        [0]
    )
    df_raw['municipio_raw'] = df_raw[col_mun].astype(str)

    # Converter colunas numéricas (vírgula → ponto, remover espaços)
    cols_num = [c for c in df_raw.columns if c not in [col_mun, 'cod_municipio', 'municipio_raw']]
    for col in cols_num:
        df_raw[col] = (
            df_raw[col]
            .astype(str)
            .str.replace('.', '', regex=False)  # separador de milhar
            .str.replace(',', '.', regex=False)
            .str.strip()
            .replace({'-': '0', '': '0'})
        )
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce').fillna(0)

    # Calcular doses agregadas por grupo etário
    def somar_cols(df, colunas):
        cols_presentes = [c for c in colunas if c in df.columns]
        return df[cols_presentes].sum(axis=1) if cols_presentes else 0

    df_out = pd.DataFrame()
    df_out['cod_municipio']     = df_raw['cod_municipio']
    df_out['ano']               = ano
    df_out['fonte']             = 'TabNet_doses'
    df_out['doses_menor1ano']   = somar_cols(df_raw, FAIXAS_MENOR1)
    df_out['doses_1a4anos']     = somar_cols(df_raw, FAIXAS_1A4)
    df_out['doses_infantil']    = somar_cols(df_raw, FAIXAS_INFANTIL)
    df_out['doses_adolescente'] = somar_cols(df_raw, FAIXAS_ADOLESC)
    df_out['doses_idoso']       = somar_cols(df_raw, FAIXAS_IDOSO)
    df_out['doses_total']       = somar_cols(df_raw, cols_num)

    return df_out[df_out['cod_municipio'].notna()].copy()


# ── Carregar todos os arquivos disponíveis ─────────────────────
anos_tabnet   = [2020, 2021, 2022, 2023]
frames_tabnet = []
anos_ok       = []
anos_falta    = []

for ano in anos_tabnet:
    # Aceita nomes como: 2022_pi.csv, pni_tabnet_PI_2022.csv, tabnet_2022.csv
    padroes = [
        f'{DADOS_BRUTOS}/{ano}_pi.csv',
        f'{DADOS_BRUTOS}/pni_tabnet_PI_{ano}.csv',
        f'{DADOS_BRUTOS}/tabnet_{ano}.csv',
        f'{DADOS_BRUTOS}/{ano}.csv',
    ]
    caminho = next((p for p in padroes if os.path.exists(p)), None)

    if caminho:
        try:
            df_tb = carregar_tabnet_doses(caminho, ano)
            df_tb = df_tb[df_tb['doses_total'] > 0]
            frames_tabnet.append(df_tb)
            anos_ok.append(ano)
            print(f"✅ {ano}: {len(df_tb)} municípios | "
                  f"doses infantil total: {df_tb['doses_infantil'].sum():,.0f}")
        except Exception as e:
            print(f"⚠️  {ano}: erro — {e}")
            anos_falta.append(ano)
    else:
        print(f"❌ {ano}: arquivo não encontrado em dados_brutos/")
        anos_falta.append(ano)

if frames_tabnet:
    df_tabnet = pd.concat(frames_tabnet, ignore_index=True)
    print(f"\n✅ TabNet consolidado: {len(df_tabnet):,} linhas | anos: {anos_ok}")
else:
    df_tabnet = pd.DataFrame()
    print("\nNenhum dado TabNet carregado.")

if anos_falta:
    print(f"\n⚠️  Anos sem dados: {anos_falta}")
    print("   Suba os CSVs em dados_brutos/ com nome AAAA_pi.csv")

✅ 2020: 224 municípios | doses infantil total: 1,020,289
✅ 2021: 224 municípios | doses infantil total: 957,673
✅ 2022: 224 municípios | doses infantil total: 1,124,719
❌ 2023: arquivo não encontrado em dados_brutos/

✅ TabNet consolidado: 672 linhas | anos: [2020, 2021, 2022]

⚠️  Anos sem dados: [2023]
   Suba os CSVs em dados_brutos/ com nome AAAA_pi.csv


In [ ]:
# ── CÉLULA 8 REVISADA: Enriquecer TabNet com nome de município ─

df_tabnet['nome_municipio'] = df_tabnet['cod_municipio'].map(MUNICIPIOS_PI)

mapeados = df_tabnet['nome_municipio'].notna().sum()
print(f"Municípios mapeados no TabNet: {mapeados} de {len(df_tabnet)}")

print("\nAmostra — Parnaíba (220770):")
print(df_tabnet[df_tabnet['cod_municipio'] == '220770'].to_string(index=False))

print("\nResumo por ano:")
print(
    df_tabnet.groupby('ano')[['doses_menor1ano','doses_1a4anos',
                               'doses_infantil','doses_total']]
    .sum()
    .applymap(lambda x: f"{x:,.0f}")
)

Municípios mapeados no TabNet: 672 de 672

Amostra — Parnaíba (220770):
cod_municipio  ano        fonte  doses_menor1ano  doses_1a4anos  doses_infantil  doses_adolescente  doses_idoso  doses_total nome_municipio
       220770 2020 TabNet_doses            13196          11226           24422               3205          458        77160       Parnaíba
       220770 2021 TabNet_doses            18573          11449           30022               2269          820        78458       Parnaíba
       220770 2022 TabNet_doses            26368          21384           47752               3329          826       119242       Parnaíba

Resumo por ano:
     doses_menor1ano doses_1a4anos doses_infantil doses_total
ano                                                          
2020         562,321       457,968      1,020,289   3,202,498
2021         549,551       408,122        957,673   2,637,058
2022         610,578       514,141      1,124,719   3,033,856


In [ ]:
# ── CÉLULA 9 REVISADA: Preparar dataset PySUS para unificação ──
# Marcar origem e selecionar colunas que existem

df['fonte'] = 'PySUS'

# Agregar PySUS: doses por município × ano (somando todas as vacinas e faixas)
df_pysus_agg = (
    df.groupby(['ano', 'cod_municipio', 'nome_municipio'], as_index=False)
    .agg(
        doses_total       = ('doses_aplicadas', 'sum'),
        cobertura_media   = ('cobertura_pct',   'mean'),
        n_registros       = ('cod_municipio',   'count'),
    )
)
df_pysus_agg['fonte'] = 'PySUS'

# Doses infantil no PySUS: faixas < 1 ano e 1–4 anos
FAIXAS_MENOR1_PYSUS = ['50','F5','G5','F6','G6','F9','G9']
FAIXAS_1A4_PYSUS    = ['51','52','53','54','F1','G1','F2','G2','F3','G3','F4','G4']
FAIXAS_INFANTIL_PYSUS = FAIXAS_MENOR1_PYSUS + FAIXAS_1A4_PYSUS

df_inf = df[df['cod_faixa'].isin(FAIXAS_INFANTIL_PYSUS)]
df_inf_agg = (
    df_inf.groupby(['ano', 'cod_municipio'], as_index=False)
    .agg(doses_infantil = ('doses_aplicadas', 'sum'))
)

df_pysus_agg = df_pysus_agg.merge(df_inf_agg, on=['ano','cod_municipio'], how='left')
df_pysus_agg['doses_infantil'] = df_pysus_agg['doses_infantil'].fillna(0)

print(f"PySUS agregado: {len(df_pysus_agg):,} linhas (ano × município)")
print(f"Período: {df_pysus_agg['ano'].min()} a {df_pysus_agg['ano'].max()}")
print(f"\nDoses infantil por ano (PySUS):")
print(df_pysus_agg.groupby('ano')['doses_infantil'].sum().apply(lambda x: f"{x:,.0f}"))

PySUS agregado: 1,120 linhas (ano × município)
Período: 2015 a 2019

Doses infantil por ano (PySUS):
ano
2015    1,461,516
2016    1,661,172
2017      939,531
2018    1,057,811
2019      260,969
Name: doses_infantil, dtype: object


In [ ]:
# ── CÉLULA 10 REVISADA: Unificar PySUS (2015–2019) + TabNet (2020–2022) ──

if len(df_tabnet) > 0:
    # Garantir que TabNet tem as mesmas colunas base
    df_tabnet_exp = df_tabnet[['ano','cod_municipio','nome_municipio',
                                'doses_total','doses_infantil','fonte']].copy()
    df_tabnet_exp['cobertura_media'] = np.nan
    df_tabnet_exp['n_registros']     = np.nan

    # Unificar
    df_unificado = pd.concat([df_pysus_agg, df_tabnet_exp], ignore_index=True)
else:
    df_unificado = df_pysus_agg.copy()
    print("⚠️  Usando apenas PySUS (2015–2019).")

# Flags analíticas (onde cobertura disponível)
df_unificado['abaixo_meta'] = df_unificado['cobertura_media'] < 95
df_unificado['critico']     = df_unificado['cobertura_media'] < 50

print(f"\n{'='*52}")
print(f"DATASET UNIFICADO")
print(f"{'='*52}")
print(f"Total de linhas:  {len(df_unificado):,}")
print(f"Período:          {df_unificado['ano'].min()} a {df_unificado['ano'].max()}")
print(f"Municípios:       {df_unificado['cod_municipio'].nunique()}")
print(f"\nLinhas por ano × fonte:")
print(df_unificado.groupby(['ano','fonte']).size().to_string())

print(f"\nDoses infantil totais por ano:")
print(
    df_unificado.groupby('ano')['doses_infantil']
    .sum()
    .apply(lambda x: f"{x:,.0f}")
    .to_string()
)

print(f"\nParnaíba (220770):")
par = df_unificado[df_unificado['cod_municipio'] == '220770']
print(par[['ano','fonte','doses_infantil','doses_total','cobertura_media']].to_string(index=False))


DATASET UNIFICADO
Total de linhas:  1,792
Período:          2015 a 2022
Municípios:       224

Linhas por ano × fonte:
ano   fonte       
2015  PySUS           224
2016  PySUS           224
2017  PySUS           224
2018  PySUS           224
2019  PySUS           224
2020  TabNet_doses    224
2021  TabNet_doses    224
2022  TabNet_doses    224

Doses infantil totais por ano:
ano
2015    1,461,516
2016    1,661,172
2017      939,531
2018    1,057,811
2019      260,969
2020    1,020,289
2021      957,673
2022    1,124,719

Parnaíba (220770):
 ano        fonte  doses_infantil  doses_total  cobertura_media
2015        PySUS         66912.0       143874        74.991364
2016        PySUS         69271.0       129176        64.610455
2017        PySUS         45998.0       117705        73.690000
2018        PySUS         46266.0       110046        68.278571
2019        PySUS          5797.0        14185         9.927500
2020 TabNet_doses         24422.0        77160              NaN
2021 

In [ ]:
# ── CÉLULA 11 REVISADA: Validação e nota metodológica ──────────

print("=" * 55)
print("RELATÓRIO DE QUALIDADE — DATASET UNIFICADO")
print("=" * 55)

total = len(df_unificado)

print("\n1. COMPLETUDE POR COLUNA")
completude = (df_unificado.notna().sum() / total * 100).round(1)
for col, pct in completude.items():
    barra = '█' * int(pct/5) + '░' * (20 - int(pct/5))
    print(f"   {col:<22} {barra} {pct:5.1f}%")

print("\n2. DOSES INFANTIL — EVOLUÇÃO TEMPORAL")
print("   (proxy de cobertura para 2020–2022)")
ev = df_unificado.groupby('ano')['doses_infantil'].sum()
base = ev.get(2019, ev.iloc[0])
for ano, doses in ev.items():
    var = f"({(doses/base-1)*100:+.1f}% vs 2019)" if ano > 2019 else ""
    print(f"   {ano}: {doses:>12,.0f} doses  {var}")

print("\n3. NOTA METODOLÓGICA")
print("""
   • 2015–2019: dados do SI-PNI via PySUS 2.3.0
     Métricas disponíveis: doses aplicadas, população-alvo,
     cobertura % (onde informada), vacina, faixa etária

   • 2020–2022: dados do TabNet DATASUS (dpnibr.def)
     Métrica disponível: doses aplicadas por faixa etária
     (todas as vacinas agregadas — sem recorte por imunobiológico)
     Faixas consideradas como 'infantil': < 1 ano a 4 anos

   • 2023: indisponível nas fontes consultadas
     Declarado como dado faltante na metodologia do TCC

   • Limitação: comparação direta de cobertura % (2015–2019)
     com doses brutas (2020–2022) requer cautela interpretativa.
     Para análise de impacto COVID, usa-se variação relativa
     de doses infantis como indicador proxy.
""")

RELATÓRIO DE QUALIDADE — DATASET UNIFICADO

1. COMPLETUDE POR COLUNA
   ano                    ████████████████████ 100.0%
   cod_municipio          ████████████████████ 100.0%
   nome_municipio         ████████████████████ 100.0%
   doses_total            ████████████████████ 100.0%
   cobertura_media        ████████████░░░░░░░░  62.5%
   n_registros            ████████████░░░░░░░░  62.5%
   fonte                  ████████████████████ 100.0%
   doses_infantil         ████████████████████ 100.0%
   abaixo_meta            ████████████████████ 100.0%
   critico                ████████████████████ 100.0%

2. DOSES INFANTIL — EVOLUÇÃO TEMPORAL
   (proxy de cobertura para 2020–2022)
   2015:    1,461,516 doses  
   2016:    1,661,172 doses  
   2017:      939,531 doses  
   2018:    1,057,811 doses  
   2019:      260,969 doses  
   2020:    1,020,289 doses  (+291.0% vs 2019)
   2021:      957,673 doses  (+267.0% vs 2019)
   2022:    1,124,719 doses  (+331.0% vs 2019)

3. NOTA METODOLÓGICA


In [18]:
# ── CÉLULA DIAGNÓSTICO: Corrigir base de comparação COVID ──────

print("=== ALERTA: 2019 com dados incompletos no PySUS ===")
print()

ev = df_unificado.groupby('ano')['doses_infantil'].sum()
for ano, doses in ev.items():
    var_2018 = f"({(doses/ev[2018]-1)*100:+.1f}% vs 2018)" if ano > 2018 else ""
    var_2019 = f"({(doses/ev[2019]-1)*100:+.1f}% vs 2019)" if ano > 2019 else ""
    flag = " ⚠️  INCOMPLETO" if ano == 2019 else ""
    print(f"  {ano}: {doses:>12,.0f} doses  {var_2018}  {flag}")

print()
print("Conclusão metodológica:")
print("  • 2019 incompleto no PySUS (apenas jan–mai aproximadamente)")
print("  • Base de comparação para impacto COVID: ANO 2018")
print("  • Queda estimada 2020 vs 2018:", f"{(ev[2020]/ev[2018]-1)*100:+.1f}%")
print("  • Queda estimada 2021 vs 2018:", f"{(ev[2021]/ev[2018]-1)*100:+.1f}%")
print("  • Recuperação 2022 vs 2018:",    f"{(ev[2022]/ev[2018]-1)*100:+.1f}%")

# Adicionar flag no dataset
df_analise['dado_incompleto'] = df_analise['ano'] == 2019
df_analise['ano_referencia_covid'] = df_analise['ano'].isin([2018])

# Salvar novamente com flag
df_analise.to_parquet(f'{DADOS_TRATADOS}/pni_piaui_unificado.parquet', index=False)
df_parnaiba = df_analise[df_analise['cod_municipio'] == '220770'].copy()
df_parnaiba.to_parquet(f'{DADOS_TRATADOS}/pni_parnaiba_unificado.parquet', index=False)

print()
print("✅ Dataset salvo com flag 'dado_incompleto' para 2019.")
print("   Use 2018 como baseline nas análises de impacto COVID.")

=== ALERTA: 2019 com dados incompletos no PySUS ===

  2015:    1,461,516 doses    
  2016:    1,661,172 doses    
  2017:      939,531 doses    
  2018:    1,057,811 doses    
  2019:      260,969 doses  (-75.3% vs 2018)   ⚠️  INCOMPLETO
  2020:    1,020,289 doses  (-3.5% vs 2018)  
  2021:      957,673 doses  (-9.5% vs 2018)  
  2022:    1,124,719 doses  (+6.3% vs 2018)  

Conclusão metodológica:
  • 2019 incompleto no PySUS (apenas jan–mai aproximadamente)
  • Base de comparação para impacto COVID: ANO 2018
  • Queda estimada 2020 vs 2018: -3.5%
  • Queda estimada 2021 vs 2018: -9.5%
  • Recuperação 2022 vs 2018: +6.3%

✅ Dataset salvo com flag 'dado_incompleto' para 2019.
   Use 2018 como baseline nas análises de impacto COVID.


In [19]:
# ── CÉLULA 12 REVISADA: Dataset analítico e salvamento ─────────

# Dataset analítico final
df_analise = df_unificado.copy()
df_analise['periodo'] = df_analise['ano'].apply(
    lambda a: 'pre_covid' if a <= 2019 else 'covid_pos'
)

# Salvar artefatos
df_analise.to_parquet(f'{DADOS_TRATADOS}/pni_piaui_unificado.parquet', index=False)

df_parnaiba = df_analise[df_analise['cod_municipio'] == '220770'].copy()
df_parnaiba.to_parquet(f'{DADOS_TRATADOS}/pni_parnaiba_unificado.parquet', index=False)

# Dataset detalhado PySUS original (com vacina, faixa, cobertura%) — para análise 03
df.to_parquet(f'{DADOS_TRATADOS}/pni_piaui_clean.parquet', index=False)

# Dicionários auxiliares
pd.DataFrame(list(MUNICIPIOS_PI.items()), columns=['cod_municipio','nome_municipio'])\
  .to_csv(f'{DADOS_BRUTOS}/municipios_piaui.csv', index=False)

pd.DataFrame(list(FAIXAS_COMPLETO.items()), columns=['cod_faixa','faixa_nome'])\
  .to_csv(f'{DADOS_BRUTOS}/dicionario_faixas_etarias.csv', index=False)

print("╔══════════════════════════════════════════════════════╗")
print("║  ✅  02_limpeza.ipynb — CONCLUÍDO                   ║")
print("╠══════════════════════════════════════════════════════╣")
artefatos = [
    (f'{DADOS_TRATADOS}/pni_piaui_unificado.parquet',    'pni_piaui_unificado.parquet'),
    (f'{DADOS_TRATADOS}/pni_parnaiba_unificado.parquet', 'pni_parnaiba_unificado.parquet'),
    (f'{DADOS_TRATADOS}/pni_piaui_clean.parquet',        'pni_piaui_clean.parquet (detalhe)'),
    (f'{DADOS_BRUTOS}/municipios_piaui.csv',             'municipios_piaui.csv'),
    (f'{DADOS_BRUTOS}/dicionario_faixas_etarias.csv',    'dicionario_faixas_etarias.csv'),
]
for caminho, nome in artefatos:
    if os.path.exists(caminho):
        tam = os.path.getsize(caminho)/1024
        print(f"║  📄 {nome:<42} {tam:5.0f} KB ║")
    else:
        print(f"║  ❌ {nome:<42} NÃO SALVO ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Período: 2015–2022 (2023 indisponível)             ║")
print(f"║  Municípios: {df_analise['cod_municipio'].nunique():<39}║")
print("╠══════════════════════════════════════════════════════╣")
print("║  PRÓXIMO: 03_analise.ipynb                          ║")
print("╚══════════════════════════════════════════════════════╝")

╔══════════════════════════════════════════════════════╗
║  ✅  02_limpeza.ipynb — CONCLUÍDO                   ║
╠══════════════════════════════════════════════════════╣
║  📄 pni_piaui_unificado.parquet                   43 KB ║
║  📄 pni_parnaiba_unificado.parquet                 7 KB ║
║  📄 pni_piaui_clean.parquet (detalhe)           3807 KB ║
║  📄 municipios_piaui.csv                           5 KB ║
║  📄 dicionario_faixas_etarias.csv                  2 KB ║
╠══════════════════════════════════════════════════════╣
║  Período: 2015–2022 (2023 indisponível)             ║
║  Municípios: 224                                    ║
╠══════════════════════════════════════════════════════╣
║  PRÓXIMO: 03_analise.ipynb                          ║
╚══════════════════════════════════════════════════════╝
